# Create Work Translation Links (oxjob #2536)

One row per candidate translation link: `origin_work_id` (the translation), `target_work_id` (the original it
translates), where it came from (`link_source`), and `served` (it passed every rule and is the one row for its edge).
`notebooks/end2end/CreateWorkLinks` unions the served rows into `openalex.works.work_links` as link type `translation`;
`/links` serves them.

What counts (dedup § 1, 2026-10-15; § 5, 2026-11-16): a translation that another journal published (its own ISSN and
DOI: a cover-to-cover English edition of a Russian journal, a Spanish journal's English Edition) is its own work,
joined to its original by this link. A second-language copy inside one journal, repository or publisher series is the
same work, which dedup merges, so it is never a translation link. The translation keeps its own byline and type.

Sources. Leg 1, what publishers declare: **Crossref** `is-translation-of` on the translation and `has-translation` on
the original (one edge either way); **DataCite** `IsTranslationOf` / `HasTranslation`. Leg 2, what oxjob #2536's
matcher found where publishers declare nothing: `openalex.works.translation_matched_pairs`, articles of a
cover-to-cover translation journal paired with their originals in the source journal by year and the whole byline,
each pair the only one for both works (a snapshot the job refreshes). A declared link wins: a matched pair is dropped
when either work already has one.

Rules (the first that fails names the drop): both ends are DOIs; both resolve (DOI locations from Crossref or DataCite,
then merges) to live works; not the same work; not the same source (a second-language copy in one journal); not two
sourceless records under one DOI prefix (one publisher's series in several languages); not declared both ways (each
record claiming to be the other's translation, so the direction is unknown). One row per edge is served. Step 5
refuses a build serving 20% fewer links than the last.

Checked before shipping (22 Nov 2026): declared, 33,100 links (Crossref 33,058, DataCite 42), almost all Russian and
Soviet journals and their English editions (Mathematical Notes, Theoretical and Mathematical Physics, Physics-Uspekhi,
Russian Mathematical Surveys, Sbornik, Izvestiya); a random 100 read by hand, 100 right. Dropped: same source 19,583,
same work 15,815, one publisher's series 6,909, original not in OpenAlex 1,514, both ways 22. Matched, 63,426 pairs in
124 Russian and Ukrainian journals and their Pleiades, Springer, IOP and SIAM editions; against the declared pairs as
ground truth it agreed on 13,133 of 13,177 (99.7%; most of the rest are the publishers' own errors), and a fresh random
100 of the new pairs read by hand was 100 right.


In [0]:
-- 1. Every declared pair, ids normalized, both directions collapsed (is-translation-of on the translation and
--    has-translation on the original say the same thing).
CREATE OR REPLACE TABLE openalex.works.work_translation_links_declared AS
WITH cr AS (
  SELECT native_id AS doi, relation.`is-translation-of` AS ito, relation.`has-translation` AS ht
  FROM openalex.crossref.crossref_deduplicated
  WHERE size(relation.`is-translation-of`) > 0 OR size(relation.`has-translation`) > 0
),
raw AS (
  SELECT 'crossref' AS src, 'doi' AS tr_ns, doi AS tr_raw, coalesce(r.`id-type`, 'doi') AS or_ns, r.id AS or_raw,
         'is-translation-of' AS decl
  FROM cr LATERAL VIEW explode(ito) t AS r
  UNION ALL
  SELECT 'crossref', coalesce(r.`id-type`, 'doi'), r.id, 'doi', doi, 'has-translation'
  FROM cr LATERAL VIEW explode(ht) t AS r
  UNION ALL
  SELECT 'datacite', 'doi', native_id, i.namespace, i.id, 'IsTranslationOf'
  FROM openalex.datacite.datacite_works LATERAL VIEW explode(ids) t AS i WHERE i.relationship = 'IsTranslationOf'
  UNION ALL
  SELECT 'datacite', i.namespace, i.id, 'doi', native_id, 'HasTranslation'
  FROM openalex.datacite.datacite_works LATERAL VIEW explode(ids) t AS i WHERE i.relationship = 'HasTranslation'
),
ns AS (
  SELECT src, decl, tr_raw, or_raw,
    CASE WHEN lower(tr_ns) = 'uri' AND lower(tr_raw) RLIKE 'doi[.]org/10[.]' THEN 'doi' ELSE lower(tr_ns) END AS tr_ns,
    CASE WHEN lower(or_ns) = 'uri' AND lower(or_raw) RLIKE 'doi[.]org/10[.]' THEN 'doi' ELSE lower(or_ns) END AS or_ns
  FROM raw
),
ids AS (
  SELECT src, decl, tr_ns, or_ns,
    CASE WHEN tr_ns = 'doi' THEN regexp_replace(lower(trim(tr_raw)), '^(https?://(dx[.])?doi[.]org/|doi:)', '')
         ELSE lower(trim(tr_raw)) END AS tr_id,
    CASE WHEN or_ns = 'doi' THEN regexp_replace(lower(trim(or_raw)), '^(https?://(dx[.])?doi[.]org/|doi:)', '')
         ELSE lower(trim(or_raw)) END AS or_id
  FROM ns
)
SELECT tr_ns, tr_id, or_ns, or_id,
       concat_ws(',', sort_array(collect_set(src))) AS declared_in,
       concat_ws(',', sort_array(collect_set(decl))) AS declared_by
FROM ids
WHERE tr_id <> '' AND or_id <> ''
GROUP BY 1, 2, 3, 4

In [0]:
-- 2. Resolve both ends to live works (DOI locations from Crossref or DataCite, then merges) and apply the rules; the
--    first failing rule names the drop.
CREATE OR REPLACE TABLE openalex.works.work_translation_links_candidates AS
WITH d AS (SELECT * FROM openalex.works.work_translation_links_declared),
ids AS (SELECT tr_id AS id FROM d WHERE tr_ns = 'doi' UNION SELECT or_id FROM d WHERE or_ns = 'doi'),
id_works AS (
  SELECT l.native_id AS id,
         MAX_BY(l.work_id, CASE l.provenance WHEN 'crossref' THEN 3 WHEN 'datacite' THEN 2 ELSE 1 END) AS work_id
  FROM openalex.works.location_work_ids l JOIN ids ON l.native_id = ids.id
  WHERE l.native_id_namespace = 'doi' AND l.provenance IN ('crossref', 'datacite') AND l.work_id IS NOT NULL
  GROUP BY 1
),
resolved AS (
  SELECT i.id, coalesce(m.merge_into_id, i.work_id) AS work_id
  FROM id_works i LEFT JOIN openalex.works.merged_work_ids_map m ON m.id = i.work_id
),
w AS (
  SELECT id, primary_location.source.id AS source_id
  FROM openalex.works.openalex_works WHERE id IN (SELECT work_id FROM resolved)
),
j AS (
  SELECT d.*, rt.work_id AS origin_work_id, ro.work_id AS target_work_id, wt.source_id AS origin_source_id,
         wo.source_id AS target_source_id, wt.id IS NOT NULL AS origin_live, wo.id IS NOT NULL AS target_live
  FROM d
  LEFT JOIN resolved rt ON d.tr_ns = 'doi' AND rt.id = d.tr_id
  LEFT JOIN resolved ro ON d.or_ns = 'doi' AND ro.id = d.or_id
  LEFT JOIN w wt ON wt.id = rt.work_id
  LEFT JOIN w wo ON wo.id = ro.work_id
),
both_ways AS (
  SELECT DISTINCT a.origin_work_id, a.target_work_id FROM j a
  JOIN j b ON a.origin_work_id = b.target_work_id AND a.target_work_id = b.origin_work_id
)
SELECT j.origin_work_id, j.target_work_id, 'declared' AS link_source, j.declared_in, j.declared_by,
       concat(j.tr_ns, ':', j.tr_id) AS translation_native_id, concat(j.or_ns, ':', j.or_id) AS original_native_id,
       CASE
         WHEN j.tr_ns <> 'doi' OR j.or_ns <> 'doi' THEN 'id type not a DOI'
         WHEN j.origin_work_id IS NULL THEN 'translation DOI not in OpenAlex'
         WHEN j.target_work_id IS NULL THEN 'original DOI not in OpenAlex'
         WHEN NOT j.origin_live THEN 'translation work not live'
         WHEN NOT j.target_live THEN 'original work not live'
         WHEN j.origin_work_id = j.target_work_id THEN 'same work'
         WHEN j.origin_source_id = j.target_source_id THEN 'same source: one work, dedup merges'
         WHEN j.origin_source_id IS NULL AND j.target_source_id IS NULL
              AND split(j.tr_id, '/')[0] = split(j.or_id, '/')[0] THEN 'one publisher series: one work, dedup merges'
         WHEN bw.origin_work_id IS NOT NULL THEN 'declared both ways'
       END AS drop_reason
FROM j LEFT JOIN both_ways bw ON bw.origin_work_id = j.origin_work_id AND bw.target_work_id = j.target_work_id

In [0]:
-- 2b. Matched pairs (leg 2): oxjob #2536's matcher pairs articles of a translation journal with their originals by
--     year and whole byline (openalex.works.translation_matched_pairs, a snapshot the job refreshes). The same hygiene
--     again (merges since the snapshot, live, same work, same source), and a matched pair yields to the publisher:
--     dropped when either work already has a declared translation link.
INSERT INTO openalex.works.work_translation_links_candidates
  (origin_work_id, target_work_id, link_source, declared_in, declared_by, translation_native_id, original_native_id,
   drop_reason)
WITH p AS (
  SELECT coalesce(m1.merge_into_id, p.translation_work_id) AS origin_work_id,
         coalesce(m2.merge_into_id, p.original_work_id) AS target_work_id
  FROM openalex.works.translation_matched_pairs p
  LEFT JOIN openalex.works.merged_work_ids_map m1 ON m1.id = p.translation_work_id
  LEFT JOIN openalex.works.merged_work_ids_map m2 ON m2.id = p.original_work_id
),
w AS (
  SELECT id, primary_location.source.id AS source_id FROM openalex.works.openalex_works
  WHERE id IN (SELECT origin_work_id FROM p UNION SELECT target_work_id FROM p)
),
dec AS (
  SELECT origin_work_id AS work_id FROM openalex.works.work_translation_links_candidates WHERE drop_reason IS NULL
  UNION SELECT target_work_id FROM openalex.works.work_translation_links_candidates WHERE drop_reason IS NULL
)
SELECT p.origin_work_id, p.target_work_id, 'matched', CAST(NULL AS STRING), CAST(NULL AS STRING),
       concat('openalex:W', p.origin_work_id), concat('openalex:W', p.target_work_id),
       CASE
         WHEN wt.id IS NULL THEN 'translation work not live'
         WHEN wo.id IS NULL THEN 'original work not live'
         WHEN p.origin_work_id = p.target_work_id THEN 'same work'
         WHEN wt.source_id = wo.source_id THEN 'same source: one work, dedup merges'
         WHEN p.origin_work_id IN (SELECT work_id FROM dec) OR p.target_work_id IN (SELECT work_id FROM dec)
           THEN 'publisher declared a translation link for this work'
       END
FROM p LEFT JOIN w wt ON wt.id = p.origin_work_id LEFT JOIN w wo ON wo.id = p.target_work_id

In [0]:
-- 3. One row per candidate; `served` marks exactly one row per (translation, original) edge that passed every rule.
--    Built beside the served table, checked, then swapped in.
CREATE OR REPLACE TABLE openalex.works.work_translation_links_build AS
WITH r AS (
  SELECT *, row_number() OVER (
    PARTITION BY origin_work_id, target_work_id
    ORDER BY drop_reason IS NULL DESC, link_source = 'declared' DESC, translation_native_id, original_native_id) AS rn
  FROM openalex.works.work_translation_links_candidates
)
SELECT origin_work_id, target_work_id, 'translation' AS link_type, link_source, declared_in, declared_by,
       (drop_reason IS NULL AND rn = 1 AND origin_work_id IS NOT NULL AND target_work_id IS NOT NULL) AS served,
       CASE WHEN drop_reason IS NOT NULL THEN drop_reason WHEN rn > 1 THEN 'duplicate edge' END AS drop_reason,
       translation_native_id, original_native_id, current_timestamp() AS built_at
FROM r

In [0]:
-- 4. Fuse: refuse a build that serves 20% fewer links than the one being replaced (a broken input, not real change).
CREATE TABLE IF NOT EXISTS openalex.works.work_translation_links LIKE openalex.works.work_translation_links_build

In [0]:
SELECT assert_true(
  (SELECT count_if(served) FROM openalex.works.work_translation_links_build)
    >= 0.8 * (SELECT count_if(served) FROM openalex.works.work_translation_links),
  'work_translation_links_build serves over 20% fewer translation links than work_translation_links: check the inputs before swapping'
) AS fuse_ok

In [0]:
-- 5. Swap in.
CREATE OR REPLACE TABLE openalex.works.work_translation_links
COMMENT 'oxjob #2536: translation links (origin = the translation, target = the original it translates); served rows go to work_links as translation'
AS SELECT * FROM openalex.works.work_translation_links_build